In [1]:
!pip install -q groq

In [2]:
import json
import functools
from google.colab import userdata
from groq import Groq, RateLimitError, AuthenticationError, APIConnectionError

In [3]:

# 1. CONNECT TO GROQ
client = Groq(api_key=userdata.get("GROQ_API_KEY"))

MODEL = "openai/gpt-oss-20b"

In [4]:
# 2. LIBRARY DATA (DICTIONARY)
BOOKS = {
    "python": {
        "author": "Mark Lutz",
        "available": 5
    },
    "java": {
        "author": "Herbert Schildt",
        "available": 3
    },
    "c programming": {
        "author": "Dennis Ritchie",
        "available": 4
    },
    "data structures": {
        "author": "Reema Thareja",
        "available": 2
    }
}

# Example dictionary lookup
print("Python book:", BOOKS.get("python", "Book not found"))

Python book: {'author': 'Mark Lutz', 'available': 5}


In [5]:
# 3. SYSTEM PROMPT
SYSTEM = f"""
ROLE:
You are a friendly assistant for a college library.

FACTS:
Available books and details:
{json.dumps(BOOKS)}

RULES:
- Answer only library-related questions.
- Never invent book availability.
- If a book is not listed, say it is not in the catalogue.
- Remember the student's name during the conversation.

FORMAT:
Give short, simple and friendly answers.
"""


In [6]:
# 4. FEW-SHOT EXAMPLES
examples = [
    {"role": "user", "content": "Do you have Python books?"},
    {"role": "assistant", "content": "Yes! We have 5 Python books available."},

    {"role": "user", "content": "Who is the author of Java?"},
    {"role": "assistant", "content": "The Java book is written by Herbert Schildt."}
]

In [8]:
# 5. FUNCTION WITH TYPE HINTS
def get_book_info(book_name: str) -> str:
    """Return the author and availability of a library book."""

    book_name = book_name.lower().strip()

    book = BOOKS.get(book_name)

    if book is None:
        return "Sorry, this book is not in our catalogue."

    return (
        f"Author: {book['author']}, "
        f"Available copies: {book['available']}"
    )

In [9]:
# 6. DECORATOR
def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"[LOG] Calling function: {func.__name__}")
        result = func(*args, **kwargs)
        print("[LOG] Function completed.")
        return result

    return wrapper


@log_call
def check_book(book_name: str) -> str:
    """Check the details of a book in the library."""
    return get_book_info(book_name)


# Test the function
print(check_book("python"))


[LOG] Calling function: check_book
[LOG] Function completed.
Author: Mark Lutz, Available copies: 5


In [10]:
# 7. JSON OUTPUT
def extract_book_request(message: str) -> dict:
    """Convert a student's message into JSON data."""

    prompt = """
    Extract the book name and request from the student's message.
    Return ONLY valid JSON with these keys:
    book_name and request.

    Example:
    Input: I want to know about Python book.
    Output: {"book_name": "python", "request": "book information"}
    """

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": message}
        ],
        temperature=0
    )

    reply = response.choices[0].message.content.strip()

    # Remove optional Markdown code fences
    if reply.startswith("```"):
        reply = reply.replace("```json", "").replace("```", "").strip()

    return json.loads(reply)


# Test JSON extraction
try:
    request_data = extract_book_request(
        "I want to know about the Python book"
    )
    print("\nExtracted JSON:")
    print(json.dumps(request_data, indent=2))

except json.JSONDecodeError:
    print("Could not convert the response into JSON.")



Extracted JSON:
{
  "book_name": "python",
  "request": "book information"
}


In [11]:

# 8. CONVERSATION MEMORY


history = [
    {"role": "system", "content": SYSTEM}
] + examples

MAX_MESSAGES = 10


def chat(user_text: str) -> str:
    """Send a message and remember the conversation."""

    history.append({
        "role": "user",
        "content": user_text
    })

    recent = [history[0]] + history[1:][-MAX_MESSAGES:]

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=recent,
            temperature=0.3
        )

        reply = response.choices[0].message.content

    except RateLimitError:
        reply = "The library assistant is busy. Please try again."

    except AuthenticationError:
        reply = "API key problem. Please check the setup."

    except APIConnectionError:
        reply = "Unable to connect. Please try again later."

    history.append({
        "role": "assistant",
        "content": reply
    })

    return reply

In [ ]:
# 9. CHATBOT LOOP


print("\nWelcome to the College Library Assistant!")
print("Type 'bye' to exit.\n")

while True:
    user_input = input("You: ")

    if user_input.lower().strip() in ["bye", "exit", "quit"]:
        print("Bot: Thank you for visiting the library!")
        break

    print("Bot:", chat(user_input))


Welcome to the College Library Assistant!
Type 'bye' to exit.

You: does python book available
Bot: Yes, we have 5 Python books available.
You: i need 6 python books
Bot: I only have 5 Python books available. May I know your name?
